# Import libraries and dataset into environment

In [1]:
import dill
import os
import sys
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import math as mt
import shap
from sklearn.metrics import confusion_matrix, roc_curve, auc
from MLstatkit import Bootstrapping
import gc
from joblib import Parallel, delayed, externals
import multiprocessing
num_cores = multiprocessing.cpu_count() - 1
import defined_functions
from defined_functions import sum_metric, met_collate_func
defined_functions.pd = pd

In [2]:
path = os.getcwd()
sys.path.append(path)

save_file = os.path.join(path, "session.pkl")

with open(save_file, "rb") as f:
    state = dill.load(f)

split_list_valid_smote = state["split_list_valid_smote"]
split_list_valid_smote_final = state["split_list_valid_smote_final"]
split_list_fil_valid_smote_final = state["split_list_fil_valid_smote_final"]
split_list_sim_onset_valid_smote_final = state["split_list_sim_onset_valid_smote_final"]
split_list_vldiag_valid_smote_final = state["split_list_vldiag_valid_smote_final"]
split_list_diag1_valid_smote_final = state["split_list_diag1_valid_smote_final"]
split_list_diag2_valid_smote_final = state["split_list_diag2_valid_smote_final"]

# Define functions

In [3]:
# Define function to train Elastic Net regression model
def run_single_elastic_split(i, split):
    train_df = split['train'].copy()
    valid_df = split['valid'].copy()
    test_df = split['test'].copy()

    train_df['outcome'] = train_df['outcome'].cat.reorder_categories(["Non severe", "Severe"], ordered = False)
    valid_df['outcome'] = valid_df['outcome'].cat.reorder_categories(["Non severe", "Severe"], ordered = False)
    test_df['outcome'] = test_df['outcome'].cat.reorder_categories(["Non severe", "Severe"], ordered = False)

    train_x = train_df.drop(columns = ['outcome'])
    valid_x = valid_df.drop(columns = ['outcome'])
    test_x = test_df.drop(columns = ['outcome'])

    features = list(train_x.columns)
    cat_cols = ['age', 'gender', 'vaccination', 'comorbidity']
    for col in cat_cols:
        train_x[col] = train_x[col].astype('category')
        valid_x[col] = valid_x[col].astype('category')
        test_x[col] = test_x[col].astype('category')

    train_x = pd.get_dummies(train_x, columns = cat_cols, drop_first = False)
    valid_x = pd.get_dummies(valid_x, columns = cat_cols, drop_first = False)
    test_x = pd.get_dummies(test_x, columns = cat_cols, drop_first = False)

    feature_names = train_x.columns.tolist()

    # Force valid/test to have same columns and same order as train
    valid_x = valid_x.reindex(columns = feature_names, fill_value = 0)
    test_x = test_x.reindex(columns = feature_names, fill_value = 0)
    
    mapping = {"Non severe": 0, "Severe": 1}
    
    train_y = train_df['outcome'].map(mapping).astype(int)
    valid_y = valid_df['outcome'].map(mapping).astype(int)
    test_y = test_df['outcome'].map(mapping).astype(int)
    elastic = LogisticRegression(
        penalty = 'elasticnet',  # Perform L1-regularization/Lasso regression
        solver = 'saga',
        l1_ratio = 0.5, # Elastic-Net mixing parameter
        C = 0.1,   # Inverse of regularization strength
        class_weight = {0: 1, 1: 1.25},    # Define wights associated with classes
        max_iter = 1000,
        random_state = 123,
        verbose = 0
    )

    elastic_model = elastic.fit(train_x, train_y)

    # Make predictions on Testing set
    elastic_pred_prob = elastic_model.predict_proba(test_x)[:, 1]

    # Calculate Area Under the ROC curve
    fpr, tpr_curve, _ = roc_curve(test_y, elastic_pred_prob, pos_label = 1)
    auc_val_elastic = auc(fpr, tpr_curve)

    # Store standard structure dictionary
    roc_container = {
        'actual': test_y,
        'probabilities': elastic_pred_prob
    }
    
    # Calculate Area Under the Precision-Recall Curve (AUPRC / PR-AUC)
    auprc_val, prc_ci_lower, prc_ci_upper = Bootstrapping(test_y, elastic_pred_prob, 'pr_auc')

    # Convert predicted probabilities to the labels
    elastic_pred = elastic_model.predict(test_x).astype(int)
    elastic_pred_res = np.where(elastic_pred == 0, "Non severe", "Severe")
    
    # Compute Confusion Matrix (Test)
    tn, fp, fn, tp = confusion_matrix(test_y, elastic_pred, labels = [0, 1]).ravel()
    
    # Format a formal R-styled evaluation matrix lookup dataframe 
    cfm_elastic = pd.DataFrame(
        [[tn, fp], [fn, tp]], 
        index = ["Non severe", "Severe"], 
        columns = ["Non severe", "Severe"]
    )
    cfm_elastic.index.name = 'Prediction'
    cfm_elastic.columns.name = 'Observed'
    
    # Calculate performance metrics
    accuracy_elastic = (tp + tn) / (tn + fp + fn + tp) if (tn + fp + fn + tp) > 0 else 0
    sensitivity_elastic = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity_elastic = tn / (tn + fp) if (tn + fp) > 0 else 0
    npv_elastic = tn / (tn + fn) if (tn + fn) > 0 else 0
    precision_elastic = tp / (tp + fp) if (tp + fp) > 0 else 0
        
    # Make predictions on Training Set to gather accuracy
    elastic_pred_train_prob = elastic_model.predict_proba(train_x)[:, 1]
    elastic_pred_train = elastic_model.predict(train_x).astype(int)
    tn_tr, fp_tr, fn_tr, tp_tr = confusion_matrix(train_y, elastic_pred_train, labels = [0, 1]).ravel()
    accuracy_elastic_train = (tp_tr + tn_tr) / (tn_tr + fp_tr + fn_tr + tp_tr)
    
    masker = shap.maskers.Independent(train_x)
    explainer = shap.LinearExplainer(model = elastic_model, masker = masker)
    shap_values = explainer(train_x).values

    return {
        "model": elastic_model,
        "confusion_matrix": cfm_elastic,
        "accuracy_training": accuracy_elastic_train,
        "accuracy_testing": accuracy_elastic,
        "sensitivity": sensitivity_elastic,
        "specificity": specificity_elastic,
        "precision": precision_elastic,
        "npv": npv_elastic,
        "roc": roc_container,
        "AUC_value": auc_val_elastic,
        "PRC_val": auprc_val,
        "PRC_lower_ci": prc_ci_lower,
        "PRC_upper_ci": prc_ci_upper,
        "SHAP_values": shap_values,
        "prediction": elastic_pred_res,
        "pred_prob": elastic_pred_prob
    }

# Define function to train Elastic Net regression for 100 times in parallel
def model_func_elastic_tune(data_list):
    
    # Count system resource availability profiles
    num_cores = multiprocessing.cpu_count() - 1
    
    if __name__ == '__main__':
        try:
            result_list = Parallel(n_jobs = num_cores)(
                delayed(run_single_elastic_split)(i, data_list[i]) 
                for i in range(100)
                )
        finally:
            externals.loky.get_reusable_executor().shutdown(wait = True)
            gc.collect()

    return result_list

# Fitting dataset into the model

## Fitting data list without VL information

In [4]:
elastic_fil_list = model_func_elastic_tune(split_list_fil_valid_smote_final)
elastic_fil_met_summary = sum_metric(elastic_fil_list)
elastic_fil_metrics_summary = elastic_fil_met_summary["metric_summary"]
elastic_fil_summary = met_collate_func(elastic_fil_metrics_summary).assign(
    models = "ElasticNet regression (No VL info & SMOTE)"
)
elastic_fil_summary

/home/blaw004/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/blaw004/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/blaw004/anaconda3/lib/python3.12/s

bound,Metrics,estimate,lower,upper,models
0,AUPRC_value,36.7492,17.3134,59.3404,ElasticNet regression (No VL info & SMOTE)
1,AUROC_value,92.1550,82.7551,97.5012,ElasticNet regression (No VL info & SMOTE)
2,Accuracy,87.0665,80.3474,93.0816,ElasticNet regression (No VL info & SMOTE)
3,Accuracy_train,87.8645,83.2386,91.8510,ElasticNet regression (No VL info & SMOTE)
4,NPV,99.4555,98.6509,100.0000,ElasticNet regression (No VL info & SMOTE)
5,Precision,17.7280,10.8492,25.9099,ElasticNet regression (No VL info & SMOTE)
6,Sensitivity,84.5000,60.0000,100.0000,ElasticNet regression (No VL info & SMOTE)
7,Specificity,87.1464,79.7352,93.9486,ElasticNet regression (No VL info & SMOTE)


## Fitting data list with simulated VL at symptom onset

In [5]:
elastic_vlsymp_sim_list = model_func_elastic_tune(split_list_sim_onset_valid_smote_final)
elastic_vlsymp_sim_met_summary = sum_metric(elastic_vlsymp_sim_list)
elastic_vlsymp_sim_metrics_summary = elastic_vlsymp_sim_met_summary["metric_summary"]
elastic_vlsymp_sim_summary = met_collate_func(elastic_vlsymp_sim_metrics_summary).assign(
    models = "ElasticNet regression (VL symp simulated & SMOTE)"
)
elastic_vlsymp_sim_summary

/home/blaw004/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/blaw004/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/blaw004/anaconda3/lib/python3.12/s

bound,Metrics,estimate,lower,upper,models
0,AUPRC_value,37.1994,17.6288,60.8743,ElasticNet regression (VL symp simulated & SMOTE)
1,AUROC_value,92.0346,82.9949,97.7188,ElasticNet regression (VL symp simulated & SMOTE)
2,Accuracy,87.0665,80.2039,92.3036,ElasticNet regression (VL symp simulated & SMOTE)
3,Accuracy_train,87.9065,83.4268,91.8017,ElasticNet regression (VL symp simulated & SMOTE)
4,NPV,99.4383,98.6441,100.0000,ElasticNet regression (VL symp simulated & SMOTE)
5,Precision,17.5079,11.6895,24.6791,ElasticNet regression (VL symp simulated & SMOTE)
6,Sensitivity,84.0000,60.0000,100.0000,ElasticNet regression (VL symp simulated & SMOTE)
7,Specificity,87.1620,80.0623,93.0140,ElasticNet regression (VL symp simulated & SMOTE)


## Fitting data list with VL at diagnosis

In [6]:
elastic_vldiag_list = model_func_elastic_tune(split_list_vldiag_valid_smote_final)
elastic_vldiag_met_summary = sum_metric(elastic_vldiag_list)
elastic_vldiag_metrics_summary = elastic_vldiag_met_summary["metric_summary"]
elastic_vldiag_summary = met_collate_func(elastic_vldiag_metrics_summary).assign(
    models = "ElasticNet regression (VL diag & SMOTE)"
)
elastic_vldiag_summary

/home/blaw004/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/blaw004/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/blaw004/anaconda3/lib/python3.12/s

bound,Metrics,estimate,lower,upper,models
0,AUPRC_value,41.2717,18.5390,63.8921,ElasticNet regression (VL diag & SMOTE)
1,AUROC_value,92.9090,83.7048,97.9634,ElasticNet regression (VL diag & SMOTE)
2,Accuracy,87.6344,80.8082,92.1601,ElasticNet regression (VL diag & SMOTE)
3,Accuracy_train,88.8583,85.0415,92.2391,ElasticNet regression (VL diag & SMOTE)
4,NPV,99.4227,98.5939,100.0000,ElasticNet regression (VL diag & SMOTE)
5,Precision,18.1155,11.9621,25.0156,ElasticNet regression (VL diag & SMOTE)
6,Sensitivity,83.5000,60.0000,100.0000,ElasticNet regression (VL diag & SMOTE)
7,Specificity,87.7632,80.5218,92.7025,ElasticNet regression (VL diag & SMOTE)


## Fitting data list with VL at diagnosis & VL at 1-day after diagnosis

In [7]:
elastic_add1_list = model_func_elastic_tune(split_list_diag1_valid_smote_final)
elastic_add1_met_summary = sum_metric(elastic_add1_list)
elastic_add1_metrics_summary = elastic_add1_met_summary["metric_summary"]
elastic_add1_summary = met_collate_func(elastic_add1_metrics_summary).assign(
    models = "ElasticNet regression (VL diag + 1 & SMOTE)"
)
elastic_add1_summary

/home/blaw004/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
Bootstrapping pr_auc:   0%|          | 0/1000 [00:00<?, ?it/s]/home/blaw004/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=N

bound,Metrics,estimate,lower,upper,models
0,AUPRC_value,41.1384,19.0248,64.4434,ElasticNet regression (VL diag + 1 & SMOTE)
1,AUROC_value,93.0988,83.7991,98.1846,ElasticNet regression (VL diag + 1 & SMOTE)
2,Accuracy,87.9033,80.9668,92.4622,ElasticNet regression (VL diag + 1 & SMOTE)
3,Accuracy_train,89.4517,86.2357,92.9491,ElasticNet regression (VL diag + 1 & SMOTE)
4,NPV,99.4350,98.6553,100.0000,ElasticNet regression (VL diag + 1 & SMOTE)
5,Precision,18.4826,12.5000,26.9143,ElasticNet regression (VL diag + 1 & SMOTE)
6,Sensitivity,83.8000,60.0000,100.0000,ElasticNet regression (VL diag + 1 & SMOTE)
7,Specificity,88.0312,80.8333,92.6869,ElasticNet regression (VL diag + 1 & SMOTE)


## Fitting data list with VL at diagnosis & VL at 2-days after diagnosis

In [8]:
elastic_add2_list = model_func_elastic_tune(split_list_diag2_valid_smote_final)
elastic_add2_met_summary = sum_metric(elastic_add2_list)
elastic_add2_metrics_summary = elastic_add2_met_summary["metric_summary"]
elastic_add2_summary = met_collate_func(elastic_add2_metrics_summary).assign(
    models = "ElasticNet regression (VL diag + 2 & SMOTE)"
)
elastic_add2_summary

/home/blaw004/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/blaw004/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/blaw004/anaconda3/lib/python3.12/s

bound,Metrics,estimate,lower,upper,models
0,AUPRC_value,42.7955,19.0450,66.5341,ElasticNet regression (VL diag + 2 & SMOTE)
1,AUROC_value,93.2847,84.1160,98.2305,ElasticNet regression (VL diag + 2 & SMOTE)
2,Accuracy,88.0634,81.7145,92.4622,ElasticNet regression (VL diag + 2 & SMOTE)
3,Accuracy_train,89.7549,86.5524,93.0724,ElasticNet regression (VL diag + 2 & SMOTE)
4,NPV,99.4501,98.6643,100.0000,ElasticNet regression (VL diag + 2 & SMOTE)
5,Precision,18.8000,12.8882,27.3392,ElasticNet regression (VL diag + 2 & SMOTE)
6,Sensitivity,84.2000,60.0000,100.0000,ElasticNet regression (VL diag + 2 & SMOTE)
7,Specificity,88.1838,81.6199,92.9984,ElasticNet regression (VL diag + 2 & SMOTE)


# Save model trained

In [ ]:
path = os.getcwd()

state = {
    "elastic_fil_list": elastic_fil_list,
    "elastic_vlsymp_sim_list": elastic_vlsymp_sim_list,
    "elastic_vldiag_list": elastic_vldiag_list,
    "elastic_add1_list": elastic_add1_list,
    "elastic_add2_list": elastic_add2_list
}

save_file = os.path.join(path, "elastic_trained.pkl")

with open(save_file, "wb") as f:
    dill.dump(state, f)

print(f"Saved to: {save_file}")

In [ ]:
path2 = os.getcwd()

state = {
    "enet_fil_summary": elastic_fil_summary,
    "enet_vlsymp_sim_summary": elastic_vlsymp_sim_summary,
    "enet_vldiag_summary": elastic_vldiag_summary,
    "enet_add2_summary": elastic_add2_summary,
    "enet_add1_summary": elastic_add1_summary
}

save_file = os.path.join(path2, "elastic_metric_summary.pkl")

with open(save_file, "wb") as f:
    dill.dump(state, f)

print(f"Saved to: {save_file}")